# Feature Engineering


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *

# Data Loading 


In [2]:
TEST=False

In [3]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


In [4]:
#just testing with two stocks for now 
if TEST==True:
    test_ticker = ['AAPL', 'NVDA']
    market_data = market_data[market_data["Ticker"].isin(test_ticker)].copy()
    filename = "test_stock_data_with_features.parquet"
else:
    filename="stock_data_with_features.parquet"
market_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
1,2016-09-29,ABBV,Healthcare,41.672211,42.660614,41.658943,42.653984,6431200
2,2016-09-29,AMZN,Consumer,41.452499,41.875000,41.231499,41.412998,98444000
3,2016-09-29,BAC,Financials,12.126826,12.398800,12.046834,12.302809,78997700
4,2016-09-29,BLK,Financials,279.353027,284.365635,276.842798,281.519152,665600
...,...,...,...,...,...,...,...,...
62820,2026-09-29,PFE,Healthcare,28.325001,28.760000,28.270000,28.600000,7477071
62821,2026-09-29,SLB,Energy,49.744999,50.770000,49.349998,50.580002,8998688
62822,2026-09-29,UNH,Healthcare,373.350006,376.589996,371.329987,374.880005,929404
62823,2026-09-29,WMT,Consumer,105.815002,107.860001,105.351997,107.470001,8665918


In [5]:
FE = FeatureEngineer(market_data, benchmark_data, filename=filename)
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400
...,...,...,...,...,...,...,...,...
62820,2026-09-23,XOM,Energy,161.229996,162.119995,159.419998,159.619995,12569300
62821,2026-09-24,XOM,Energy,162.139999,164.910004,161.199997,163.059998,12484900
62822,2026-09-25,XOM,Energy,160.589996,161.789993,159.779999,160.229996,10996700
62823,2026-09-28,XOM,Energy,162.520004,163.539993,161.050003,162.800003,12809500


Benchmark Features

In [6]:
list(benchmark_data["Ticker"].unique())

['QQQ', 'SPY']

# Returns for the benchmark

In [7]:
FE.add_benchmark_returns()


FE.data


Rows: 62825
Duplicate ticker/date pairs: 0


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425
...,...,...,...,...,...,...,...,...,...,...
62820,2026-09-23,XOM,Energy,161.229996,162.119995,159.419998,159.619995,12569300,-0.008362,-0.007202
62821,2026-09-24,XOM,Energy,162.139999,164.910004,161.199997,163.059998,12484900,-0.000148,-0.000821
62822,2026-09-25,XOM,Energy,160.589996,161.789993,159.779999,160.229996,10996700,0.004588,0.005435
62823,2026-09-28,XOM,Energy,162.520004,163.539993,161.050003,162.800003,12809500,-0.010705,-0.007441


In [8]:
print("Rows:", len(FE.data))

print(
    "Duplicate ticker/date pairs:",
    FE.data.duplicated(
        ["Ticker", "Date"]
    ).sum()
)

print(
    FE.data.groupby("Ticker")["Date"]
    .nunique()
    .describe()
)

Rows: 62825
Duplicate ticker/date pairs: 0
count      25.0
mean     2513.0
std         0.0
min      2513.0
25%      2513.0
50%      2513.0
75%      2513.0
max      2513.0
Name: Date, dtype: float64


# Returns

In [8]:
FE.add_daily_returns() #Daily Price Movement 
FE.add_return_feature() #Very short term momentum - default is 5 days. 
FE.add_return_feature(10)
FE.add_return_feature(20)
FE.add_return_feature(60)

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Return_10D,Return_20D,Return_60D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,0.007755,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,-0.004688,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,0.004266,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,0.000442,NaN,NaN,NaN,NaN


# Volatility Features

In [9]:

FE.add_volatility_feature() #recent variability - default is 20 days
FE.add_volatility_feature(5) 
FE.add_volatility_feature(10) #recent variability - default is 20 days
FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Return_10D,Return_20D,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,0.007755,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,-0.004688,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,0.004266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,0.000442,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Trend Features

In [10]:
FE.add_SMA_feature() #recent price trend - default is 20 
FE.add_SMA_feature(50) #longer term price trend 
FE.add_price_relative_to_sma() #position relative to trend - default is 20
FE.add_price_relative_to_sma(50) #position relative to longer trend 

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_10D,Return_20D,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Volume Features

In [11]:
FE.add_average_volume_feature() #Trading activity - default is 20

FE.add_relative_volume()

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62820,2026-09-23,XOM,Energy,161.229996,162.119995,159.419998,159.619995,12569300,-0.008362,-0.007202,...,0.192665,0.017563,0.017737,0.019043,162.077502,158.323460,0.994771,1.018358,14801000.00,0.849220
62821,2026-09-24,XOM,Energy,162.139999,164.910004,161.199997,163.059998,12484900,-0.000148,-0.000821,...,0.193607,0.017205,0.018112,0.019024,162.275002,158.694654,0.999168,1.021711,14683325.00,0.850277
62822,2026-09-25,XOM,Energy,160.589996,161.789993,159.779999,160.229996,10996700,0.004588,0.005435,...,0.186013,0.017151,0.018354,0.019028,162.482501,159.006233,0.988353,1.009960,14518735.00,0.757414
62823,2026-09-28,XOM,Energy,162.520004,163.539993,161.050003,162.800003,12809500,-0.010705,-0.007441,...,0.193175,0.017313,0.009815,0.019585,162.773001,159.328394,0.998446,1.020032,14591580.00,0.877869


# Target Variable - 5 Day return

In [13]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-10-04,AAPL,Technology,25.841478,26.141056,25.756864,25.855199,118947200,-0.001518,-0.005098,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-05,AAPL,Technology,25.852909,25.992407,25.770582,25.932949,85812400,0.003548,0.004425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62820,2026-09-23,XOM,Energy,161.229996,162.119995,159.419998,159.619995,12569300,-0.008362,-0.007202,...,0.192665,0.017563,0.017737,0.019043,162.077502,158.323460,0.994771,1.018358,14801000.00,0.849220
62821,2026-09-24,XOM,Energy,162.139999,164.910004,161.199997,163.059998,12484900,-0.000148,-0.000821,...,0.193607,0.017205,0.018112,0.019024,162.275002,158.694654,0.999168,1.021711,14683325.00,0.850277
62822,2026-09-25,XOM,Energy,160.589996,161.789993,159.779999,160.229996,10996700,0.004588,0.005435,...,0.186013,0.017151,0.018354,0.019028,162.482501,159.006233,0.988353,1.009960,14518735.00,0.757414
62823,2026-09-28,XOM,Energy,162.520004,163.539993,161.050003,162.800003,12809500,-0.010705,-0.007441,...,0.193175,0.017313,0.009815,0.019585,162.773001,159.328394,0.998446,1.020032,14591580.00,0.877869


In [14]:
FE.add_target_variable(5)


FE.data = FE.data.dropna().copy()

FE.data.index
FE.save_features()

FE.data

Features data saved for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
1     2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
2     2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
3     2016-10-04   AAPL  Technology   25.841478   26.141056   25.756864   
4     2016-10-05   AAPL  Technology   25.852909   25.992407   25.770582   
...          ...    ...         ...         ...         ...         ...   
62820 2026-09-23    XOM      Energy  161.229996  162.119995  159.419998   
62821 2026-09-24    XOM      Energy  162.139999  164.910004  161.199997   
62822 2026-09-25    XOM      Energy  160.589996  161.789993  159.779999   
62823 2026-09-28    XOM      Energy  162.520004  163.539993  161.050003   
62824 2026-09-29    XOM      Energy  161.625000  161.639999  159.270004   

Price        Open     Volume  
0       25.878078  143548000  
1       25.71

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D
60,2016-12-23,AAPL,Technology,26.783262,26.783262,26.569493,26.569493,56998000.0,0.000666,0.001464,...,0.007573,0.004581,0.006500,26.066441,25.910281,1.027500,1.033692,123165640.0,0.462775,-0.003175
61,2016-12-27,AAPL,Technology,26.953356,27.077480,26.776363,26.783258,73187600.0,0.005158,0.002481,...,0.007566,0.004736,0.005955,26.131836,25.911342,1.031438,1.040215,121386220.0,0.602932,-0.010575
62,2016-12-28,AAPL,Technology,26.838427,27.128049,26.709704,27.013119,83623600.0,-0.007780,-0.008264,...,0.007681,0.005150,0.004217,26.192749,25.910471,1.024651,1.035814,119861640.0,0.697668,-0.001285
63,2016-12-29,AAPL,Technology,26.831528,26.918874,26.755674,26.767166,60158000.0,-0.001418,-0.000223,...,0.007284,0.005105,0.004227,26.264121,25.909827,1.021604,1.035573,115637080.0,0.520231,0.010109
64,2016-12-30,AAPL,Technology,26.622362,26.939568,26.532717,26.813146,122345200.0,-0.010275,-0.003655,...,0.007157,0.005479,0.004825,26.336872,25.906601,1.010840,1.027629,114336960.0,1.070041,0.027370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62815,2026-09-16,XOM,Energy,163.320007,167.500000,162.559998,167.229996,13437800.0,0.000255,-0.004410,...,0.016501,0.022358,0.018198,162.861002,156.272881,1.002818,1.045095,13751285.0,0.977203,-0.012797
62816,2026-09-17,XOM,Energy,163.270004,163.389999,161.449997,161.830002,10976500.0,0.017312,0.011339,...,0.016472,0.022038,0.018186,162.786002,156.733840,1.002973,1.041702,13617665.0,0.806049,-0.006921
62817,2026-09-18,XOM,Energy,163.539993,163.539993,161.309998,162.500000,37640600.0,0.006319,0.001287,...,0.016353,0.021850,0.017739,162.655502,157.273127,1.005438,1.039847,14809545.0,2.541645,-0.018038
62818,2026-09-21,XOM,Energy,158.300003,161.949997,157.940002,160.960007,15752500.0,0.028821,0.015505,...,0.017772,0.025582,0.019943,162.315002,157.679397,0.975264,1.003936,14881455.0,1.058532,0.026658


In [15]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D
60,2016-12-23,AAPL,Technology,26.783262,26.783262,26.569493,26.569493,56998000.0,0.000666,0.001464,...,0.007573,0.004581,0.006500,26.066441,25.910281,1.027500,1.033692,123165640.0,0.462775,-0.003175
61,2016-12-27,AAPL,Technology,26.953356,27.077480,26.776363,26.783258,73187600.0,0.005158,0.002481,...,0.007566,0.004736,0.005955,26.131836,25.911342,1.031438,1.040215,121386220.0,0.602932,-0.010575
62,2016-12-28,AAPL,Technology,26.838427,27.128049,26.709704,27.013119,83623600.0,-0.007780,-0.008264,...,0.007681,0.005150,0.004217,26.192749,25.910471,1.024651,1.035814,119861640.0,0.697668,-0.001285
63,2016-12-29,AAPL,Technology,26.831528,26.918874,26.755674,26.767166,60158000.0,-0.001418,-0.000223,...,0.007284,0.005105,0.004227,26.264121,25.909827,1.021604,1.035573,115637080.0,0.520231,0.010109
64,2016-12-30,AAPL,Technology,26.622362,26.939568,26.532717,26.813146,122345200.0,-0.010275,-0.003655,...,0.007157,0.005479,0.004825,26.336872,25.906601,1.010840,1.027629,114336960.0,1.070041,0.027370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62815,2026-09-16,XOM,Energy,163.320007,167.500000,162.559998,167.229996,13437800.0,0.000255,-0.004410,...,0.016501,0.022358,0.018198,162.861002,156.272881,1.002818,1.045095,13751285.0,0.977203,-0.012797
62816,2026-09-17,XOM,Energy,163.270004,163.389999,161.449997,161.830002,10976500.0,0.017312,0.011339,...,0.016472,0.022038,0.018186,162.786002,156.733840,1.002973,1.041702,13617665.0,0.806049,-0.006921
62817,2026-09-18,XOM,Energy,163.539993,163.539993,161.309998,162.500000,37640600.0,0.006319,0.001287,...,0.016353,0.021850,0.017739,162.655502,157.273127,1.005438,1.039847,14809545.0,2.541645,-0.018038
62818,2026-09-21,XOM,Energy,158.300003,161.949997,157.940002,160.960007,15752500.0,0.028821,0.015505,...,0.017772,0.025582,0.019943,162.315002,157.679397,0.975264,1.003936,14881455.0,1.058532,0.026658


In [16]:
data = FE.load_features()

data.columns

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
1     2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
2     2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
3     2016-10-04   AAPL  Technology   25.841478   26.141056   25.756864   
4     2016-10-05   AAPL  Technology   25.852909   25.992407   25.770582   
...          ...    ...         ...         ...         ...         ...   
62820 2026-09-23    XOM      Energy  161.229996  162.119995  159.419998   
62821 2026-09-24    XOM      Energy  162.139999  164.910004  161.199997   
62822 2026-09-25    XOM      Energy  160.589996  161.789993  159.779999   
62823 2026-09-28    XOM      Energy  162.520004  163.539993  161.050003   
62824 2026-09-29    XOM      Energy  161.625000  161.639999  159.270004   

Price        Open     Volume  
0       25.878078  143548000  
1       25.717984  14

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D', 'Return_10D',
       'Return_20D', 'Return_60D', 'Volatility_20D', 'Volatility_5D',
       'Volatility_10D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D',
       'Future_Return_5D'],
      dtype='str', name='Price')

In [17]:


data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D
60,2016-12-23,AAPL,Technology,26.783262,26.783262,26.569493,26.569493,56998000.0,0.000666,0.001464,...,0.007573,0.004581,0.006500,26.066441,25.910281,1.027500,1.033692,123165640.0,0.462775,-0.003175
61,2016-12-27,AAPL,Technology,26.953356,27.077480,26.776363,26.783258,73187600.0,0.005158,0.002481,...,0.007566,0.004736,0.005955,26.131836,25.911342,1.031438,1.040215,121386220.0,0.602932,-0.010575
62,2016-12-28,AAPL,Technology,26.838427,27.128049,26.709704,27.013119,83623600.0,-0.007780,-0.008264,...,0.007681,0.005150,0.004217,26.192749,25.910471,1.024651,1.035814,119861640.0,0.697668,-0.001285
63,2016-12-29,AAPL,Technology,26.831528,26.918874,26.755674,26.767166,60158000.0,-0.001418,-0.000223,...,0.007284,0.005105,0.004227,26.264121,25.909827,1.021604,1.035573,115637080.0,0.520231,0.010109
64,2016-12-30,AAPL,Technology,26.622362,26.939568,26.532717,26.813146,122345200.0,-0.010275,-0.003655,...,0.007157,0.005479,0.004825,26.336872,25.906601,1.010840,1.027629,114336960.0,1.070041,0.027370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62815,2026-09-16,XOM,Energy,163.320007,167.500000,162.559998,167.229996,13437800.0,0.000255,-0.004410,...,0.016501,0.022358,0.018198,162.861002,156.272881,1.002818,1.045095,13751285.0,0.977203,-0.012797
62816,2026-09-17,XOM,Energy,163.270004,163.389999,161.449997,161.830002,10976500.0,0.017312,0.011339,...,0.016472,0.022038,0.018186,162.786002,156.733840,1.002973,1.041702,13617665.0,0.806049,-0.006921
62817,2026-09-18,XOM,Energy,163.539993,163.539993,161.309998,162.500000,37640600.0,0.006319,0.001287,...,0.016353,0.021850,0.017739,162.655502,157.273127,1.005438,1.039847,14809545.0,2.541645,-0.018038
62818,2026-09-21,XOM,Energy,158.300003,161.949997,157.940002,160.960007,15752500.0,0.028821,0.015505,...,0.017772,0.025582,0.019943,162.315002,157.679397,0.975264,1.003936,14881455.0,1.058532,0.026658


In [18]:
data.isna().sum()

Price
Date                   0
Ticker                 0
Sector                 0
Close                  0
High                   0
Low                    0
Open                   0
Volume                 0
QQQ_Return             0
SPY_Return             0
Daily_Return           0
Return_5D              0
Return_10D             0
Return_20D             0
Return_60D             0
Volatility_20D         0
Volatility_5D          0
Volatility_10D         0
SMA_20D                0
SMA_50D                0
Price_to_SMA_20D       0
Price_to_SMA_50D       0
Volume_Average_20D     0
Relative_Volume_20D    0
Future_Return_5D       0
dtype: int64